####E-commerce Business Analytics using Advanced SQL

####Data Exploration

####Objective

Explore the e-commerce datasets to understand:
- Customer distribution
- Order behavior
- Sales and revenue patterns
- Payment behavior
- Product categories
- Time-based trends
- Relationships between business entities

The exploration phase is performed before data cleaning and data modeling.

#### Order Status Analysis

In [0]:
%sql
SELECT
    order_status,
    COUNT(*) AS order_count
FROM raw_orders
GROUP BY order_status
ORDER BY order_count DESC;

#####Observation

The dataset contains 99,441 orders across 8 order statuses.
The majority of orders are marked as delivered, while the remaining orders are distributed across shipped, canceled, unavailable, invoiced, processing, created, and approved statuses.

This provides an initial understanding of the order lifecycle and will be useful for further analysis of sales and delivery performance.

####Orders Over Time

In [0]:
%sql
SELECT
    DATE_FORMAT(
        TO_TIMESTAMP(order_purchase_timestamp, 'dd-MM-yyyy HH:mm'),
        'yyyy-MM'
    ) AS order_month,
    COUNT(*) AS order_count
FROM raw_orders
GROUP BY
    DATE_FORMAT(
        TO_TIMESTAMP(order_purchase_timestamp, 'dd-MM-yyyy HH:mm'),
        'yyyy-MM'
    )
ORDER BY order_month;

####Observation – Monthly Order Volume

Order volume varies over time, with a noticeable increase during 2017 and 2018.
November 2017 recorded 7,544 orders, while January 2018 recorded 7,269 orders.

The September and October 2018 values are significantly lower because the dataset contains only a small number of records for the end of the observation period. Therefore, these partial months should not be directly compared with complete months.

####Revenue Analysis

In [0]:
%sql
SELECT
    DATE_FORMAT(
        TO_TIMESTAMP(o.order_purchase_timestamp, 'dd-MM-yyyy HH:mm'),
        'yyyy-MM'
    ) AS order_month,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM raw_orders o
JOIN raw_order_items oi
    ON o.order_id = oi.order_id
GROUP BY
    DATE_FORMAT(
        TO_TIMESTAMP(o.order_purchase_timestamp, 'dd-MM-yyyy HH:mm'),
        'yyyy-MM'
    )
ORDER BY order_month;

####Observation – Monthly Revenue

Monthly revenue generally increased during 2017 and remained at a relatively high level during the first half of 2018.

November 2017 recorded revenue of approximately 1.01 million, while April and May 2018 were close to 1 million.

The very low revenue observed in September 2018 is associated with the partial period represented in the dataset and should not be compared directly with complete months.

####Average Order Value

In [0]:
%sql

SELECT
    DATE_FORMAT(
        TO_TIMESTAMP(o.order_purchase_timestamp, 'dd-MM-yyyy HH:mm'),
        'yyyy-MM'
    ) AS order_month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(
        SUM(oi.price) / COUNT(DISTINCT o.order_id),
        2
    ) AS average_order_value
FROM raw_orders o
JOIN raw_order_items oi
    ON o.order_id = oi.order_id
GROUP BY
    DATE_FORMAT(
        TO_TIMESTAMP(o.order_purchase_timestamp, 'dd-MM-yyyy HH:mm'),
        'yyyy-MM'
    )
ORDER BY order_month;

####Observation – Average Order Value

Average Order Value (AOV) varies across months, generally remaining within a relatively consistent range during 2017.

The AOV was approximately 152.49 in January 2017 and decreased to around 125.48 in July 2017. It increased again in September and October 2017.

The very high or low AOV values in the early months of the dataset should be interpreted carefully because those months contain a small number of orders.

####Top Product Categories by Revenue

In [0]:
%sql

SELECT
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM raw_order_items oi
JOIN raw_products p
    ON oi.product_id = p.product_id
WHERE p.product_category_name IS NOT NULL
GROUP BY p.product_category_name
ORDER BY total_revenue DESC
LIMIT 10;

#####Observation – Top Product Categories by Revenue

The analysis shows that beleza_saude generated the highest revenue among the product categories, followed by relogios_presentes and cama_mesa_banho.

The results demonstrate that revenue contribution differs significantly across product categories.

This analysis can help identify categories that contribute substantially to overall business revenue.

####Top Customers by Revenue

In [0]:
%sql

SELECT
    o.customer_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM raw_orders o
JOIN raw_order_items oi
    ON o.order_id = oi.order_id
GROUP BY o.customer_id
ORDER BY total_revenue DESC
LIMIT 10;

####Observation – Top Customers by Revenue

The highest-revenue customers generated significantly more revenue than the average customer.

However, each of the top 10 customers placed only one order, indicating that their high revenue was driven by expensive purchases rather than frequent purchases.

This suggests that large-value transactions contribute substantially to revenue for certain customers.

####Repeat Customer Analysis

In [0]:
%sql

SELECT
    CASE
        WHEN total_orders = 1 THEN 'One-Time Customer'
        ELSE 'Repeat Customer'
    END AS customer_type,
    COUNT(*) AS customer_count
FROM (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS total_orders
    FROM raw_orders o
    JOIN raw_customers c
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
) t
GROUP BY customer_type
ORDER BY customer_type;

##### Observation – Repeat Customer Analysis

The analysis identifies 93,099 one-time customers and 2,997 repeat customers using customer_unique_id.

This shows that most customers placed an order only once, while a smaller group made purchases across multiple orders.

customer_unique_id was used instead of customer_id because it represents the unique customer across the dataset.

####Payment Method Analysis

In [0]:
%sql

SELECT
    payment_type,
    COUNT(*) AS payment_transactions,
    ROUND(SUM(payment_value), 2) AS total_payment_value,
    ROUND(AVG(payment_value), 2) AS average_payment_value
FROM raw_order_payments
GROUP BY payment_type
ORDER BY total_payment_value DESC;

####Observation – Payment Method Analysis

Credit card is the most frequently recorded payment method and also accounts for the largest total payment value.

Boleto is the second-largest payment method by both transaction count and total payment value.

Voucher transactions have a lower average payment value compared with the other defined payment methods.

The dataset also contains 3 not_defined payment records with a payment value of 0.

####Delivery Performance Analysis

In [0]:
%sql

SELECT
    ROUND(
        AVG(
            DATEDIFF(
                TO_TIMESTAMP(order_delivered_customer_date, 'dd-MM-yyyy HH:mm'),
                TO_TIMESTAMP(order_purchase_timestamp, 'dd-MM-yyyy HH:mm')
            )
        ),
        2
    ) AS average_delivery_days
FROM raw_orders
WHERE order_delivered_customer_date IS NOT NULL;

####Observation – Delivery Performance

The average delivery time from order purchase to customer delivery is approximately 12.5 days for orders with a recorded delivery date.

This metric provides a baseline for evaluating the overall delivery performance of the e-commerce business.

####Estimated vs Actual Delivery

In [0]:
%sql

SELECT
    COUNT(*) AS late_deliveries
FROM raw_orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
  AND TO_TIMESTAMP(order_delivered_customer_date, 'dd-MM-yyyy HH:mm')
      > TO_TIMESTAMP(order_estimated_delivery_date, 'dd-MM-yyyy HH:mm');

####Calculate delivery performance rate

In [0]:
%sql

SELECT
    COUNT(*) AS eligible_deliveries,
    SUM(
        CASE
            WHEN TO_TIMESTAMP(order_delivered_customer_date, 'dd-MM-yyyy HH:mm')
                 > TO_TIMESTAMP(order_estimated_delivery_date, 'dd-MM-yyyy HH:mm')
            THEN 1
            ELSE 0
        END
    ) AS late_deliveries,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN TO_TIMESTAMP(order_delivered_customer_date, 'dd-MM-yyyy HH:mm')
                     > TO_TIMESTAMP(order_estimated_delivery_date, 'dd-MM-yyyy HH:mm')
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS late_delivery_rate,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN TO_TIMESTAMP(order_delivered_customer_date, 'dd-MM-yyyy HH:mm')
                     <= TO_TIMESTAMP(order_estimated_delivery_date, 'dd-MM-yyyy HH:mm')
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS on_time_delivery_rate
FROM raw_orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;

####Observation – Delivery Performance

Among 96,476 deliveries with both actual and estimated delivery dates available, 7,827 deliveries were completed after the estimated delivery date.

The late-delivery rate was 8.11%, while the on-time delivery rate was 91.89%.

This provides a useful baseline for evaluating delivery performance and identifying potential areas for operational improvement.

####Customer Geography Analysis

In [0]:
%sql

SELECT
    customer_state,
    COUNT(DISTINCT customer_unique_id) AS customer_count
FROM raw_customers
GROUP BY customer_state
ORDER BY customer_count DESC;

####Observation – Customer Geography

The customer base is concentrated in a small number of states.

SP has the largest customer base with 40,302 customers, followed by RJ with 12,384 and MG with 11,259.

The remaining states have considerably smaller customer counts compared with these three states.

This geographic distribution can help the business understand where its customer base is concentrated and where further market analysis may be useful.

####Revenue by Customer State

In [0]:
%sql

SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM raw_customers c
JOIN raw_orders o
    ON c.customer_id = o.customer_id
JOIN raw_order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_state
ORDER BY total_revenue DESC;